In [ ]:
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "training"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and training/ at or above {Path.cwd()}")

# The `beta_ols` package lives next to this notebook in `train-notebooks/beta-training/`
BETA_TRAINING_DIR = CODE_DIR / "train-notebooks" / "beta-training"
if not (BETA_TRAINING_DIR / "beta_ols").is_dir():
    raise RuntimeError(f"`beta_ols` package not found in {BETA_TRAINING_DIR}")

# Run from `code/` and expose `code/`, `code/kev/`, and the beta-training folder on sys.path
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(BETA_TRAINING_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from dataclasses import replace

from models.kev_pointer import KevPointerModel
from training import free_gpu

import beta_ols
from beta_ols import core as beta_core
from beta_train.config import TrainConfig, scale_lr_for_sample_size
from beta_train.diagnostics import find_max_micro_chunk, gradient_check
from beta_train.items import build_items_cached, items_summary
from beta_train.model_state import reset_trainable, snapshot_trainable
from beta_train.reports import best_beta_table, best_table, comparison_table, eval_beta_table, mse_decomposition, per_dataset_table
from beta_train.runs import load_best
from beta_train.trainer import RunResult, load_run, run_training
from beta_train.validation import PopulationValidator, TrainingPathValidator, validate_all_cached, validation_metrics_frame

# Disk cache for training items and the untrained (step-0) validation; delete the folder to clear it
CACHE_DIR = CODE_DIR / ".cache" / "beta_training"

print(f"CODE_DIR: {CODE_DIR}")

In [ ]:
# Require CUDA and release any previously loaded `kev_model` before allocating a fresh 800M backbone
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required to run jaredpalmer/kev-0.8b in bf16 train mode")
if "kev_model" in globals():
    globals().pop("kev_model")
device = "cuda"
free_gpu()
torch.manual_seed(42)

# Model settings (also the key of the cached step-0 validation)
KEV_BASE = "jaredpalmer/kev-0.8b"
LORA_RANK = 64
WEIGHTS_DTYPE = "bf16"
HEAD_TEMPERATURE = 1.0   # training uses raw uncalibrated temperature = 1.0

# Build the frozen backbone + LoRA + pointer head
t_model0 = time.perf_counter()
kev_model = KevPointerModel.build(
    KEV_BASE,
    device=device,
    lora_rank=LORA_RANK,
    weights_dtype=WEIGHTS_DTYPE,
)
kev_model.net.head.temperature = HEAD_TEMPERATURE
kev_model.train()                     # enable training mode (LoRA dropout active, trainable params require_grad)
torch.cuda.synchronize()
model_load_sec = time.perf_counter() - t_model0

trainable_params_count = sum(p.numel() for p in kev_model.trainable_parameters())
total_params_count = sum(p.numel() for p in kev_model.parameters())
display(
    pd.Series(
        {
            "base_model": KEV_BASE,
            "mode": kev_model.mode,
            "training": kev_model.training,
            "weights_dtype": kev_model.meta.weights_dtype,
            "lora_rank": kev_model.meta.lora,
            "trainable_params": f"{trainable_params_count:,}",
            "total_params": f"{total_params_count:,}",
            "model_load_sec": round(model_load_sec, 3),
        },
        name="kev_800m_train_mode",
    ).to_frame()
)

# Snapshot the untrained trainable weights right after building, so every run, diagnostic, and the step-0
# validation start from them (rebuilding the model re-snapshots)
INIT_TRAINABLE_STATE = snapshot_trainable(kev_model)
MODEL_KEY = {"base": KEV_BASE, "lora_rank": LORA_RANK, "weights_dtype": WEIGHTS_DTYPE, "head_temperature": HEAD_TEMPERATURE}
print(f"Snapshotted {len(INIT_TRAINABLE_STATE)} trainable tensors")

# Datasets (loaded and replicated through `beta_ols`)

In [ ]:
# Datasets this notebook uses; only these are loaded (the others are skipped to save time)
ITEM_DATASETS = [         # training items are built for these (training, gradient check, VRAM search)
    "wu2025",
    "berke2023",
    "grummon2023_calories",   # Grummon 2023, beta target = calories of the chosen product (printed label)
    "grummon2023_labels",     # Grummon 2023, beta target = product choice on the label it shows
]
VALIDATION_DATASETS = [   # validation runs on these
    "grummon2025",
    "kriner2026",
]
LOADED_DATASETS = list(ITEM_DATASETS)
for dataset_name in VALIDATION_DATASETS:
    if dataset_name not in LOADED_DATASETS:
        LOADED_DATASETS.append(dataset_name)

# Load each through its `beta_ols` module: task rows, regression rows, regression specs, human OLS replication.
# Validation uses the whole dataset. Training splits a dataset with several task variables (`v`) into one pseudo
# dataset per task ("<dataset>/<v>"), so an item holds one respondent's tasks of a single kind.
choice_dfs: dict[str, pd.DataFrame] = {}
regression_rows_by_dataset: dict[str, pd.DataFrame] = {}
regression_specs_by_dataset: dict[str, dict] = {}
human_ols_by_dataset: dict[str, pd.DataFrame] = {}
module_by_dataset = {}    # (pseudo) dataset name -> the beta_ols module that codes its rows
TRAIN_DATASETS = []       # pseudo datasets of ITEM_DATASETS, the names training uses

for dataset_name in LOADED_DATASETS:
    dataset_module = beta_ols.DATASET_MODULES[dataset_name]
    full_choice_df = dataset_module.load_choice_df()
    pieces = {}
    if dataset_name in VALIDATION_DATASETS:
        pieces[dataset_name] = full_choice_df
    if dataset_name in ITEM_DATASETS:
        task_pieces = beta_core.split_by_task(full_choice_df, dataset_name)
        pieces.update(task_pieces)
        TRAIN_DATASETS.extend(task_pieces)

    for piece_name, choice_df in pieces.items():
        rows = dataset_module.regression_rows(choice_df)
        specs = dataset_module.regression_specs(rows)
        module_by_dataset[piece_name] = dataset_module
        choice_dfs[piece_name] = choice_df
        regression_rows_by_dataset[piece_name] = rows
        regression_specs_by_dataset[piece_name] = specs
        human_ols_by_dataset[piece_name] = beta_core.fit_human_ols(rows, specs)
        print(f"{piece_name}: {choice_df['CASEID'].nunique():,} respondents | {len(choice_df):,} tasks | {len(specs)} regression(s)")
        display(beta_core.wide_ols_table(human_ols_by_dataset[piece_name]).round(4))
print(f"TRAIN_DATASETS = {TRAIN_DATASETS}")


# Model, validators, and training items

In [ ]:
# One validator per VALIDATION_DATASETS entry (population OLS from the dataset's module); pick how inference runs
VALIDATOR_KIND = "fused"   # "fused": LoRA merged + fused kernels + CUDA graphs, persona states shared
                           # "training_path": the training forward in eval mode, LoRA unmerged (no merge/offload/capture)
# "fused" per-GPU-batch inference budgets: lower them if validation runs out of memory (bigger models), raise for speed
VALIDATION_MAX_QUESTIONS = 1024
VALIDATION_MAX_CHARS = 1_000_000
# "training_path" items (data rows) per forward pass (no gradients, so far above the training micro-chunk)
VALIDATION_EVAL_CHUNK = 32

validators = {}
if VALIDATOR_KIND == "fused":
    for dataset_name in VALIDATION_DATASETS:
        validators[dataset_name] = PopulationValidator(
            beta_ols.DATASET_MODULES[dataset_name],
            choice_dfs[dataset_name],
            regression_specs_by_dataset[dataset_name],
            max_questions=VALIDATION_MAX_QUESTIONS,
            max_chars=VALIDATION_MAX_CHARS,
        )
elif VALIDATOR_KIND == "training_path":
    # The training path runs on items (tokenized respondents), cached on disk like the training items
    validation_modules = {}
    for dataset_name in VALIDATION_DATASETS:
        validation_modules[dataset_name] = beta_ols.DATASET_MODULES[dataset_name]
    validation_items, validation_items_report = build_items_cached(kev_model, validation_modules, choice_dfs, cache_dir=CACHE_DIR)
    display(validation_items_report)
    for dataset_name in VALIDATION_DATASETS:
        validators[dataset_name] = TrainingPathValidator(
            beta_ols.DATASET_MODULES[dataset_name],
            choice_dfs[dataset_name],
            regression_specs_by_dataset[dataset_name],
            validation_items[dataset_name],
            eval_chunk=VALIDATION_EVAL_CHUNK,
        )
else:
    raise ValueError(f"VALIDATOR_KIND must be 'fused' or 'training_path', got {VALIDATOR_KIND!r}")
display(pd.DataFrame([validator.describe() for validator in validators.values()]).set_index("validator"))

In [ ]:
# Untrained (step-0) validation, loaded from the disk cache when it already ran for this model, data, and code;
# reused as the first validation row of every run
step0_validation, step0_cache_report = validate_all_cached(
    kev_model,
    validators,
    INIT_TRAINABLE_STATE,
    model_key=MODEL_KEY,
    cache_dir=CACHE_DIR,
)
display(step0_cache_report)
display(validation_metrics_frame(step0_validation).round(4))
for dataset_name, (_, beta_df) in step0_validation.items():
    display(beta_df.round(4))

In [ ]:
# Tokenize the TRAIN_DATASETS rows into training items (one per data row), loaded from the disk cache when they were already
# built for this data, tokenizer, and code; Kev's prepared targets are checked against each respondent's beta part

item_modules = {}
for dataset_name in TRAIN_DATASETS:
    item_modules[dataset_name] = module_by_dataset[dataset_name]
unified_items, items_cache_report = build_items_cached(kev_model, item_modules, choice_dfs, cache_dir=CACHE_DIR)
display(items_cache_report)
display(items_summary(unified_items, regression_specs_by_dataset))

# Find max micro chunk

In [ ]:
# Largest micro-chunk (items = data rows per chunk) that fits in VRAM for the worst-case (most padded) chunks of these datasets
SEARCH_GRADIENT_CHECKPOINTING = False   # search with the setting you will train with
SEARCH_MAX_CHUNK = 64
SEARCH_CANDIDATES_PER_SIZE = 3          # worst-case chunks tested per size; a size fits only if all of them fit

search_items = []
for dataset_name in TRAIN_DATASETS:
    search_items.extend(unified_items[dataset_name])
micro_chunk_search_df, MAX_FITTING_MICRO_CHUNK = find_max_micro_chunk(
    kev_model,
    search_items,
    INIT_TRAINABLE_STATE,
    max_chunk=SEARCH_MAX_CHUNK,
    gradient_checkpointing=SEARCH_GRADIENT_CHECKPOINTING,
    candidates_per_size=SEARCH_CANDIDATES_PER_SIZE,
)
display(micro_chunk_search_df)
print(f"gradient_checkpointing={SEARCH_GRADIENT_CHECKPOINTING} | MAX_FITTING_MICRO_CHUNK = {MAX_FITTING_MICRO_CHUNK}")

# Train

In [ ]:
# Run the search above first (Grummon 2023 is new to training); keep the smaller of its result and v8's Wu + Berke value 16
MAX_FITTING_MICRO_CHUNK = min(MAX_FITTING_MICRO_CHUNK, 16)
SEARCH_GRADIENT_CHECKPOINTING = False
print(f"MAX_FITTING_MICRO_CHUNK = {MAX_FITTING_MICRO_CHUNK}")

In [ ]:
# Run folders: one per run under RUNS_DIR (stats, betas per validation, checkpoints/best.pt, checkpoints/last.pt)
NOTEBOOK = "train-beta-v9-wu-berke-grummon-800m"
RUNS_DIR = BETA_TRAINING_DIR / "runs" / NOTEBOOK
# ONE validator's column picks checkpoints/best.pt: "kriner2026/val_loss" = Kriner's KL + beta_lambda * std beta MSE
# (the run's own training loss form). Grummon 2025 is not used for selection: its values at the best step are a held-out
# check. Other choices: any "kriner2026/<metric>" column, e.g. "kriner2026/beta_mse" ("min"), "kriner2026/spearman_rho" ("max").
BEST_METRIC = "kriner2026/val_loss"
BEST_MODE = "min"

# Base run settings; each run cell below sets the sample size (tasks = training items per step, the same for every
# training dataset; with it, the peak LR and the validation steps) and beta_lambda. Training datasets keep their
# natural mix: each contributes ceil(its tasks / S) samples per epoch.
BETA_LAMBDA = 1.0
NUM_EPOCHS = 1
# Peak LR = BASE_LR_PER_TASK x sqrt(sample size in tasks = training items per step), for any mix of training datasets.
# The base is v4's 2e-5 per respondent converted to tasks: v4's items were whole Wu respondents (10 tasks each), so
# 2e-5 x sqrt(S / 10) = (2e-5 / sqrt(10)) x sqrt(S); S = 160 tasks = v4's S = 16 respondents at LR 8e-5
BASE_LR_PER_TASK = 2e-5 / math.sqrt(10)
# Rows (training items) seen between validations, per validator, so every sample size validates at the same points in
# the epoch (all validators also run at step 0 and the last step). 1,280 / 6,400 rows = every 20 / 100 steps at S = 64.
# kriner2026 picks best.pt, so it runs often; grummon2025 is the held-out check (best.pt is validated on it at the end
# if it did not run at that step)
EVAL_EVERY_ROWS_BY_VALIDATOR = {"kriner2026": 1280, "grummon2025": 6400}
BASE_SAMPLE_SIZE = 40   # v8's run to compare against: S = 40, lambda = 1

# Training sets (dataset names -> their pseudo datasets in TRAIN_DATASETS). Both have Wu + Berke and the same Grummon 2023
# respondents and choices; they differ only in Grummon 2023's beta target. Never train on both codings at once (that
# would count Grummon 2023's KL twice).
TRAIN_SET_DATASETS = {
    "gr23-calories": ["wu2025", "berke2023", "grummon2023_calories"],
    "gr23-choice": ["wu2025", "berke2023", "grummon2023_labels"],
}
TRAIN_SETS = {}
for train_set, dataset_names in TRAIN_SET_DATASETS.items():
    pseudo_datasets = []
    for dataset_name in dataset_names:
        dataset_pieces = []
        for train_dataset in TRAIN_DATASETS:
            if module_by_dataset[train_dataset].NAME == dataset_name:
                dataset_pieces.append(train_dataset)
        if len(dataset_pieces) == 0:
            raise ValueError(f"{train_set}: no training items for {dataset_name!r}; add it to ITEM_DATASETS")
        pseudo_datasets.extend(dataset_pieces)
    TRAIN_SETS[train_set] = pseudo_datasets
    print(f"{train_set}: {pseudo_datasets}")

BASE_TRAIN_SET = "gr23-calories"
BASE_CONFIG = TrainConfig(
    train_datasets=tuple(TRAIN_SETS[BASE_TRAIN_SET]),
    sample_size_by_dataset={ts: BASE_SAMPLE_SIZE for ts in TRAIN_SETS[BASE_TRAIN_SET]},
    beta_lambda=BETA_LAMBDA,
    lr=scale_lr_for_sample_size(BASE_LR_PER_TASK, BASE_SAMPLE_SIZE),
    num_epochs=NUM_EPOCHS,
    eval_every_by_validator={"kriner2026": 20, "grummon2025": 100},
    lr_schedule="onecycle",
    weight_decay=0.01,
    micro_chunk=MAX_FITTING_MICRO_CHUNK,
    gradient_checkpointing=SEARCH_GRADIENT_CHECKPOINTING,
    grad_clip=1.0,
    seed=42,
)


In [ ]:
def make_run_config(train_set: str, sample_size: int, beta_lambda: float) -> TrainConfig:
    # BASE_CONFIG with this training set (a key of TRAIN_SETS), this beta_lambda, this sample size for every training
    # dataset, its sqrt-scaled peak LR, and validation steps = rows between validations / S
    lr = scale_lr_for_sample_size(BASE_LR_PER_TASK, sample_size)
    eval_every_by_validator = {}
    for validator_name, eval_every_rows in EVAL_EVERY_ROWS_BY_VALIDATOR.items():
        eval_every_by_validator[validator_name] = max(1, round(eval_every_rows / sample_size))
    train_datasets = TRAIN_SETS[train_set]
    sample_size_by_dataset = {}
    for dataset_name in train_datasets:
        sample_size_by_dataset[dataset_name] = sample_size
    return replace(
        BASE_CONFIG,
        train_datasets=tuple(train_datasets),
        beta_lambda=beta_lambda,
        sample_size_by_dataset=sample_size_by_dataset,
        lr=lr,
        eval_every_by_validator=eval_every_by_validator,
    )


def run_name(train_set: str, config: TrainConfig) -> str:
    # Run folder name from the settings this notebook varies: training set, lambda, peak LR, sample size (rows per
    # step), epochs
    if list(config.train_datasets) != TRAIN_SETS[train_set]:
        raise ValueError(f"Config trains on {config.train_datasets}, not training set {train_set!r}")
    sample_sizes = sorted(set(config.as_dict()["sample_size_by_dataset"].values()))
    sample_size_text = "-".join(str(size) for size in sample_sizes)
    return f"{train_set}_lambda={config.beta_lambda:g}_lr={config.lr:g}_S={sample_size_text}_ep={config.num_epochs}"

# Settings recorded in each run's run.json (not used by training)
RUN_METADATA = {
    "notebook": NOTEBOOK,
    "model": MODEL_KEY,
    "item_datasets": ITEM_DATASETS,
    "train_sets": TRAIN_SET_DATASETS,
    "validation_datasets": VALIDATION_DATASETS,
    "validator_kind": VALIDATOR_KIND,
    "eval_every_rows_by_validator": EVAL_EVERY_ROWS_BY_VALIDATOR,
    "base_lr_per_task": BASE_LR_PER_TASK,
}
runs: dict[str, RunResult] = {}
display(pd.Series(BASE_CONFIG.as_dict(), name="base_config").to_frame())

In [ ]:
# Run 1: Wu + Berke + Grummon 2023 with its calorie target; S = 40 tasks per step, beta_lambda = 1 (v8's settings)
train_set = "gr23-calories"
sample_size = 40
beta_lambda = 1.0
run_config = make_run_config(train_set, sample_size, beta_lambda)
run_dir = RUNS_DIR / run_name(train_set, run_config)
runs[run_dir.name] = run_training(
    model=kev_model,
    items_by_dataset=unified_items,
    specs_by_dataset=regression_specs_by_dataset,
    validators=validators,
    step0_validation=step0_validation,
    init_snapshot=INIT_TRAINABLE_STATE,
    config=run_config,
    device=device,
    run_dir=run_dir,
    best_metric=BEST_METRIC,
    best_mode=BEST_MODE,
    run_metadata=RUN_METADATA,
    from_scratch=False,   # an existing run is resumed (or loaded, if finished); True deletes it and trains again
)

In [ ]:
# Run 2: Wu + Berke + Grummon 2023 with its label-choice target; S = 40 tasks per step, beta_lambda = 1 (v8's settings)
train_set = "gr23-choice"
sample_size = 40
beta_lambda = 1.0
run_config = make_run_config(train_set, sample_size, beta_lambda)
run_dir = RUNS_DIR / run_name(train_set, run_config)
runs[run_dir.name] = run_training(
    model=kev_model,
    items_by_dataset=unified_items,
    specs_by_dataset=regression_specs_by_dataset,
    validators=validators,
    step0_validation=step0_validation,
    init_snapshot=INIT_TRAINABLE_STATE,
    config=run_config,
    device=device,
    run_dir=run_dir,
    best_metric=BEST_METRIC,
    best_mode=BEST_MODE,
    run_metadata=RUN_METADATA,
    from_scratch=False,   # an existing run is resumed (or loaded, if finished); True deletes it and trains again
)

In [ ]:
# Both runs next to v8's run with the same settings (S = 40, lambda = 1, Wu + Berke only)
V8_RUN_DIR = BETA_TRAINING_DIR / "runs" / "train-beta-v8-wu-berke-sample-lrscale-800m" / "lambda=1_lr=4e-05_S=40_ep=1"
v8_vs_v9_runs = {"v8 wu+berke": load_run(V8_RUN_DIR)}
for train_set in TRAIN_SETS:
    v9_run_dir = RUNS_DIR / run_name(train_set, make_run_config(train_set, 40, 1.0))
    v8_vs_v9_runs[f"v9 {train_set}"] = load_run(v9_run_dir)
display(best_table(v8_vs_v9_runs).round(5))
for dataset_name in validators:
    display(best_beta_table(v8_vs_v9_runs, step0_validation, dataset_name).round(4))

In [ ]:
# Compare finished runs: final validation metrics, per-dataset train losses, and each validator's betas
display(comparison_table(runs, step0_validation).round(5))
display(per_dataset_table(runs).round(5))
for dataset_name in validators:
    display(eval_beta_table(runs, step0_validation, dataset_name).round(4))

# The same runs at their best validation (by each run's best metric)
display(best_table(runs).round(5))
for dataset_name in validators:
    display(best_beta_table(runs, step0_validation, dataset_name).round(4))

In [ ]:
# Load every saved run of this notebook from disk (no training; works in a fresh kernel after the setup cells)
runs = {}
for saved_run_dir in sorted(RUNS_DIR.iterdir()):
    runs[saved_run_dir.name] = load_run(saved_run_dir)
display(pd.DataFrame([run.summary for run in runs.values()]).set_index("label"))

In [ ]:
# Put one run's best checkpoint into kev_model (e.g. to evaluate it further)
BEST_TRAIN_SET = "gr23-choice"
BEST_RUN_DIR = RUNS_DIR / run_name(BEST_TRAIN_SET, make_run_config(BEST_TRAIN_SET, BASE_SAMPLE_SIZE, BETA_LAMBDA))
best_checkpoint = load_best(BEST_RUN_DIR)
reset_trainable(kev_model, best_checkpoint["trainable"])
display(pd.Series(best_checkpoint["best"], name=BEST_RUN_DIR.name).to_frame())

In [ ]:
# Decompose each validator's beta MSE into per-beta squared errors across runs
for dataset_name in validators:
    display(mse_decomposition(runs, step0_validation, dataset_name).round(5))

# Cross-dataset beta scale and sample-size target noise

In [ ]:
# Compare raw vs sigma_y-standardized human beta mean-square across the loaded datasets
scale_rows = []
for dataset_name, tidy in human_ols_by_dataset.items():
    choice_df = choice_dfs[dataset_name]
    option_counts = sorted({len(one_hot) for one_hot in choice_df["one_hot"]})
    sigma_by_model = tidy.groupby("model", sort=False)["sigma_y"].first()
    scale_rows.append(
        {
            "dataset": dataset_name,
            "respondents": choice_df["CASEID"].nunique(),
            "tasks": len(choice_df),
            "options_per_task": ", ".join(str(n) for n in option_counts),
            "regressions": len(sigma_by_model),
            "num_betas": len(tidy),
            "sigma_y": ", ".join(f"{sigma:.4g}" for sigma in sigma_by_model),
            "raw_signal_beta_ms": float((tidy["human_beta"] ** 2).mean()),
            "std_signal_beta_ms": float((tidy["std_beta"] ** 2).mean()),
        }
    )
display(pd.DataFrame(scale_rows).set_index("dataset").round(5))

# Rank deficiency

In [ ]:
# Target noise of sample betas vs row sample size (Monte Carlo over data rows = training items), per regression
for SNR_DATASET in TRAIN_DATASETS:  
    SNR_SAMPLE_SIZES = [16, 32, 64, 128, 256]
    SNR_DRAWS = 4000

    beta_noise_df = beta_core.beta_noise_by_sample_size(
        regression_rows_by_dataset[SNR_DATASET],
        regression_specs_by_dataset[SNR_DATASET],
        SNR_SAMPLE_SIZES,
        SNR_DRAWS,
        seed=42,
    )
    display(beta_noise_df.set_index(["model", "sample_size"]).round(4))

# Test micro-batching

In [ ]:
# Two-pass micro-chunked gradient (training) vs one backward through the whole sample
CHECK_DATASET = "grummon2023_calories/bev_choice"   # any key of `unified_items`
CHECK_SAMPLE_SIZE = 15          # one graph of this many items (rows) must fit in VRAM (method A)
CHECK_MICRO_CHUNK = 3           # items (data rows) per chunk
CHECK_BETA_LAMBDA = 5.0         # > 0 so both KL and beta gradients are tested

gradient_check_result = gradient_check(
    kev_model,
    unified_items[CHECK_DATASET],
    regression_specs_by_dataset[CHECK_DATASET],
    INIT_TRAINABLE_STATE,
    sample_size=CHECK_SAMPLE_SIZE,
    micro_chunk=CHECK_MICRO_CHUNK,
    beta_lambda=CHECK_BETA_LAMBDA,
    seed=0,
    device=device,
)
display(pd.Series(gradient_check_result, name="gradient_check").to_frame())

# Overfit first

In [ ]:
# Overfit sanity check: many passes with pure KL/CE, constant LR, and zero weight decay
BASE_LR_SAMPLE_1 = 2e-5   # validated 1-respondent/step LR; scaled by sqrt(sample size)

OVERFIT_CONFIG = TrainConfig(
    train_datasets=("grummon2023_calories/bev_choice",),
    sample_size_by_dataset={"grummon2023_calories/bev_choice": 128},
    beta_lambda=0.0,
    lr=scale_lr_for_sample_size(BASE_LR_SAMPLE_1, 128),
    num_epochs=10,
    eval_every_by_validator={name: math.ceil(len(unified_items["grummon2023_calories/bev_choice"]) / 128) for name in VALIDATION_DATASETS},   # one validation per epoch
    lr_schedule="constant",
    weight_decay=0.0,
    micro_chunk=MAX_FITTING_MICRO_CHUNK,
    gradient_checkpointing=SEARCH_GRADIENT_CHECKPOINTING,
    grad_clip=1.0,
    seed=42,
)
overfit_run = run_training(
    model=kev_model,
    items_by_dataset=unified_items,
    specs_by_dataset=regression_specs_by_dataset,
    validators=validators,
    step0_validation=step0_validation,
    init_snapshot=INIT_TRAINABLE_STATE,
    config=OVERFIT_CONFIG,
    device=device,
    run_dir=RUNS_DIR / f"overfit_{run_name(OVERFIT_CONFIG)}",
    best_metric=BEST_METRIC,
    best_mode=BEST_MODE,
    run_metadata=RUN_METADATA,
    from_scratch=False,   # an existing run is resumed (or loaded, if finished); True deletes it and trains again
)